<a href="https://colab.research.google.com/github/andreagrioni/Tutorials/blob/master/Filter_Bed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Filter BED notebook

short script to extend boundaries of a GTF file to arbitrary size and intersect it with user BED files stored into GitHub repo.

## Mount Google Drive

No more required.

In [0]:
# from google.colab import drive
# drive.mount('/content/drive')

## install dependency, download data and annotation

In [7]:
! rm -rf Tutorials
! git clone https://github.com/andreagrioni/Tutorials.git
! apt-get install bedtools
! mkdir results
#! mkdir annotation
#! wget ftp://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_32/gencode.v32.primary_assembly.annotation.gtf.gz -P annotation/

Cloning into 'Tutorials'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 140 (delta 1), reused 0 (delta 0), pack-reused 132
Receiving objects: 100% (140/140), 7.04 MiB | 10.00 MiB/s, done.
Resolving deltas: 100% (67/67), done.
Reading package lists... Done
Building dependency tree       
Reading state information... Done
bedtools is already the newest version (2.26.0+dfsg-5).
The following package was automatically installed and is no longer required:
  libnvidia-common-430
Use 'apt autoremove' to remove it.
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.
mkdir: cannot create directory ‘results’: File exists
mkdir: cannot create directory ‘annotation’: File exists
--2020-01-14 15:23:15--  ftp://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_32/gencode.v32.primary_assembly.annotation.gtf.gz
           => ‘annotation/gencode.v32.primary_assembly.annotation.gtf.gz.2’

In [13]:
! ls Tutorials/data/filter_bed

hotspots.unstranded.threshold.0.15.bed	hotspots.unstranded.threshold.0.65.bed
hotspots.unstranded.threshold.0.1.bed	hotspots.unstranded.threshold.0.6.bed
hotspots.unstranded.threshold.0.25.bed	hotspots.unstranded.threshold.0.75.bed
hotspots.unstranded.threshold.0.2.bed	hotspots.unstranded.threshold.0.7.bed
hotspots.unstranded.threshold.0.35.bed	hotspots.unstranded.threshold.0.85.bed
hotspots.unstranded.threshold.0.3.bed	hotspots.unstranded.threshold.0.8.bed
hotspots.unstranded.threshold.0.45.bed	hotspots.unstranded.threshold.0.95.bed
hotspots.unstranded.threshold.0.4.bed	hotspots.unstranded.threshold.0.9.bed
hotspots.unstranded.threshold.0.55.bed	README.md
hotspots.unstranded.threshold.0.5.bed	refseq_grch38.zip


Archive:  Tutorials/data/filter_bed/refseq_grch38.zip
replace refseq_grch38.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: refseq_grch38.tsv       


## Filter annotation for UTR
Filtered files with candidates mapping within the first 1000bp from TSS.

In [26]:
! unzip Tutorials/data/filter_bed/refseq_grch38.zip
! tail -n +2 refseq_grch38.tsv | awk ' BEGIN{OFS="\t"}{if($4=="+") {print $3,$5,$5+1000,$2 "_" $13,".",$4} else {print $3,$6-1000,$6,$2 "_" $13,".",$4} }' > refseq_hg38_UCSC_TSS.bed
! more refseq_hg38_UCSC_TSS.bed

Archive:  Tutorials/data/filter_bed/refseq_grch38.zip
replace refseq_grch38.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: refseq_grch38.tsv       
chr1	67108072	67109072	XM_011541469.1_C1orf141	.	-
chr1	67130183	67131183	XM_011541467.1_C1orf141	.	-
chr1	67130227	67131227	XM_017001276.1_C1orf141	.	-
chr1	67133962	67134962	XM_011541465.2_C1orf141	.	-
chr1	67133971	67134971	NR_075077.1_C1orf141	.	-
chr1	67133971	67134971	NM_001276352.1_C1orf141	.	-
chr1	67133971	67134971	NM_001276351.1_C1orf141	.	-
chr1	67140646	67141646	XM_011541466.2_C1orf141	.	-
chr1	67130227	67131227	XM_017001277.1_C1orf141	.	-
chr1	67130227	67131227	XM_011541473.2_C1orf141	.	-
chr1	67130183	67131183	XM_011541472.1_C1orf141	.	-
chr1	201283451	201284451	NM_001005337.2_PKP1	.	+
chr1	201283451	201284451	NM_000299.3_PKP1	.	+
chr1	8422687	8423687	NM_001042682.1_RERE	.	-
chr1	8525035	8526035	XM_005263466.1_RERE	.	-
chr1	8702474	8703474	XM_005263464.2_RERE	.	-
chr1	8736558	8737558	XM_017001359.1_RERE	.	-
chr1	881

In [0]:
def resize_annotation(infile, strand=True):
  with open(infile, 'r') as f, open("refseq_hg38_UCSC_TSS.bed", "w") as output:
    for index, line in enumerate(f.readlines()):
      if index == 0:
        continue
      column = line.strip().split("\t")
      if column[3] == "+":
        start = column[4]
        end = int(column[4]) + 1000
      elif column[3] == "-":
        start = int(column[5]) - 1000
        end = int(column[5])
        if start < 0:
          start = 0
      else:
        print(line)
      string_out = f'{column[2]}\t{start}\t{end}\t{column[1]}_{column[12]}\t.\t{column[3]}\n'
      output.write(string_out)
        

## Intersect with BedTools

In [0]:
import subprocess


def run_bedtools(candidate_file, annotation, outdir):

  candidate_file_name = os.path.basename(candidate_file).replace('.bed', '.filtered.tss_1kb.bed')
  outname = f'{outdir}/{candidate_file_name}'
  cmd = f'bedtools intersect -a {candidate_file} -b {annotation} -u > {outname}'
  print(cmd)
  try:
    subprocess.run(cmd, shell=True)
  except:
    return "error"

## run code

In [38]:
import os

resize_annotation( "refseq_grch38.tsv", strand=True)

annotation_path = "refseq_hg38_UCSC_TSS.bed"
bed_dir_path = "Tutorials/data/filter_bed/"
output_dir_path = "results/"

for bed_file in os.listdir(bed_dir_path):
  if not bed_file.endswith('.bed'):
    continue
  candidates_bed_path = os.path.join(bed_dir_path, bed_file)
  print(candidates_bed_path)
  run_bedtools(candidates_bed_path, annotation_path, output_dir_path)


Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.4.bed
bedtools intersect -a Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.4.bed -b refseq_hg38_UCSC_TSS.bed -u > results//hotspots.unstranded.threshold.0.4.filtered.tss_1kb.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.1.bed
bedtools intersect -a Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.1.bed -b refseq_hg38_UCSC_TSS.bed -u > results//hotspots.unstranded.threshold.0.1.filtered.tss_1kb.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.9.bed
bedtools intersect -a Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.9.bed -b refseq_hg38_UCSC_TSS.bed -u > results//hotspots.unstranded.threshold.0.9.filtered.tss_1kb.bed
Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.55.bed
bedtools intersect -a Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.55.bed -b refseq_hg38_UCSC_TSS.bed -u > results//hotspots.unstranded.threshold.0.55.filtered.tss_1kb.bed
Tutorials

In [29]:
! bedtools intersect -a Tutorials/data/filter_bed/hotspots.unstranded.threshold.0.1.bed -b refseq_hg38_UCSC_TSS.bed -u > results//hotspots.unstranded.threshold.0.1.filtered.tss_1kb.be

Error: Invalid record in file refseq_hg38_UCSC_TSS.bed. Record is 
chr7_KZ208912v1_fix	-611	389	NM_001322263.2_LOC100507507	.	-


## Download Results

This section allows the user to download results

In [39]:
from google.colab import files
from zipfile import ZipFile


with ZipFile('./results_tss_filtered.zip', 'w') as zipObj:
  for bed_file in os.listdir(output_dir_path):
    bed_path = os.path.join(output_dir_path, bed_file)
    print(bed_path)
    zipObj.write(bed_path)

files.download('./results_tss_filtered.zip')

results/hotspots.unstranded.threshold.0.2.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.1.filtered.tss_1kb.be
results/hotspots.unstranded.threshold.0.3.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.15.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.7.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.35.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.55.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.1.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.25.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.95.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.5.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.9.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.45.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.4.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.6.filtered.tss_1kb.bed
results/hotspots.unstranded.threshold.0.65.filtere